In [ ]:
!pip install torchinfo
!pip install torchmetrics

In [ ]:
import os
# ------------------------ for the splitting of dataset ------------------------
import random
from pathlib import Path
# ------------------------for the lebling of dataset ------------------------
from torchvision import datasets, transforms
from torch.utils.data import DataLoader #turn the dataset into iterable and also for batching
from PIL import Image
from typing import List
# ------------------------ ------------------------
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models  # ------------------
from sklearn.metrics import precision_score, recall_score, f1_score
# ------------------------ for plotting the curves ------------------------
import matplotlib.pyplot as plt
# ------------------------ for getting the summary of the model ------------------------
from torchinfo import summary

import json # To save metrics

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Section 1

In [ ]:
# walk through the dataset directory and print number of images in each subdirectory

def walk_through_dir(dir_path):
    for dirpath, dirname, filenames in os.walk(dir_path):
        print(f"There are {len(dirname)} directories and {len(filenames)} images in '{dirpath}'")


dataset_path = "/kaggle/input/f-k-mri-clean"
walk_through_dir(dataset_path)  #& pass the dataset path here ...

In [ ]:
train_dir = os.path.join(dataset_path, "train")
val_dir   = os.path.join(dataset_path, "val")

In [ ]:
from PIL import Image

train_transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert("RGB")),   # <--- ensures 3 channels always
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),

    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


val_transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225]),
])

In [ ]:
train_dataset = datasets.ImageFolder(train_dir, transform = train_transform)
val_dataset   = datasets.ImageFolder(val_dir, transform = val_transform)

print("Original class mapping:", train_dataset.class_to_idx)
print("---------------------------------------------------------")
print (train_dataset)
print("=====================================================")
print (val_dataset)

In [ ]:
batch_size = 32
n_workers = 0

train_loader = DataLoader(train_dataset,batch_size=batch_size, shuffle=True, num_workers = n_workers)

val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of validation samples: {len(val_dataset)}")

print("============================================")

print(f"length of train_loader : {len(train_loader)} batches of  {batch_size}")
print(f"length of val_loader   : {len(val_loader)} batches of  {batch_size}")

# Section #2

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

num_classes = 1  # binary classification
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False


model.fc = nn.Sequential(
    nn.Flatten(),
    nn.Dropout(0.3),
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, num_classes)
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


In [ ]:
from torchinfo import summary
summary(model, input_size=(1, 3, 224, 224))

In [ ]:
from torchsummary import summary
summary(model, (3, 224, 224))

In [ ]:
# Save the model temporarily
torch.save(model.state_dict(), "temp_model.pth")

# Get file size in MB
size_MB = os.path.getsize("temp_model.pth") / (1024 * 1024)
print(f"Model size: {size_MB:.2f} MB")

# Optional: remove the temporary file
os.remove("temp_model.pth")


In [ ]:
criterion = nn.BCEWithLogitsLoss()  
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
from timeit import default_timer as timer
def print_train_time (start : float,
                      end : float,
                      device : torch.device = None):

    """ print diffrence between start and end time. """

    total_time = end - start
    print(f"Total training time on {device}: {total_time:.3f} seconds")
    return total_time




In [ ]:
dummy_input = torch.randn(4, 3, 224, 224).to(device)
print(model(dummy_input).shape)  # Should be: torch.Size([4, 1]) to indecate binary classification

In [ ]:
print("Original class mapping:", train_dataset.class_to_idx)

In [ ]:
from tqdm.auto import tqdm  # for progress bar

torch.manual_seed(42)
start_time = timer() # start timing
pos = 1
num_epochs = 20

#?---------------------------- save metrics ----------------------------
train_losses, val_losses = [], []
train_accuracies, val_accuracies = [], []
train_precisions, train_recalls, train_f1s = [], [], []
val_precisions, val_recalls, val_f1s = [], [], []
#%---------------------------- Training Loop ----------------------------
for epoch in range(num_epochs):
    model.train() #$the model use the batchnorm and dropout layers

    running_loss, correct, total = 0.0, 0, 0

    all_labels, all_preds = [], []


    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]"):
        inputs, labels = inputs.to(device), labels.to(device)

#----------------------- train step -----------------------
        outputs = model(inputs) # &----------------------- forward pass

        labels = labels.float().unsqueeze(1)
        loss = criterion(outputs, labels) #& ------------- compute the loss (predicytions vs true labels

        optimizer.zero_grad() #& --------------------------clear the gradients, set it to zero so it start fresh for this batch/ loop
        loss.backward() #& ------------------------------- backward pass
        optimizer.step() #& ------------------------------update the weights

    #= collect loss & metrics

        probs = torch.sigmoid(outputs) #convert logits to probabilities
        predicted = (probs > 0.5).long()   #convert probabilities to binary

        running_loss += loss.item() * inputs.size(0)
        correct += (predicted == labels.long()).sum().item()
        total += labels.size(0)

#?----------------------- collect the predictions and true labels

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(predicted.cpu().numpy()) #collect true lebles and prediction across all batches

#*----------------------- compute metrics for training (storing them in the lists) -----------------------
    train_loss = running_loss / len(train_dataset)
    train_acc = correct / total
    train_precision = precision_score(all_labels, all_preds, average='binary', pos_label=pos)
    train_recall = recall_score(all_labels, all_preds, average='binary', pos_label=pos)
    train_f1 = f1_score(all_labels, all_preds, average='binary', pos_label=pos)

    train_losses.append(train_loss)
    train_accuracies.append(train_acc)
    train_precisions.append(train_precision)
    train_recalls.append(train_recall)
    train_f1s.append(train_f1)

    #$ --------------------------------------------- Validation Loop ---------------------------------------------
    model.eval() #$---------------------------- note : the model.eval disable dropout and batchnorm layers
    val_running_loss, val_correct, val_total = 0.0,0 ,0
    val_labels_all, val_preds_all = [], []

    with torch.no_grad():

        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]"):
            inputs, labels = inputs.to(device), labels.to(device)


            #only calculate the forward pass and the loss for the validation set

            outputs = model(inputs)

            labels = labels.float().unsqueeze(1)  # shape [B,1], Adds a new dimension of size 1, to model outputs,which usually are [batch_size, 1] for binary classification.
            loss = criterion(outputs, labels)

            probs = torch.sigmoid(outputs)
            predicted = (probs > 0.5).long()

            val_running_loss += loss.item() * inputs.size(0)
            val_correct += (predicted == labels.long()).sum().item()
            val_total += labels.size(0)

            val_labels_all.extend(labels.cpu().numpy())
            val_preds_all.extend(predicted.cpu().numpy())

    val_loss = val_running_loss / len(val_dataset)
    val_acc = val_correct / val_total
    val_precision = precision_score(val_labels_all, val_preds_all, average='binary', pos_label=pos)
    val_recall = recall_score(val_labels_all, val_preds_all, average='binary', pos_label=pos)
    val_f1 = f1_score(val_labels_all, val_preds_all, average='binary', pos_label=pos)

    val_losses.append(val_loss)
    val_accuracies.append(val_acc)
    val_precisions.append(val_precision)
    val_recalls.append(val_recall)
    val_f1s.append(val_f1)

       # Save metrics after each epoch
    metrics = {
        "train_losses": train_losses,
        "val_losses": val_losses,
        "train_accuracies": train_accuracies,
        "val_accuracies": val_accuracies,
        "train_precisions": train_precisions,
        "train_recalls": train_recalls,
        "train_f1s": train_f1s,
        "val_precisions": val_precisions,
        "val_recalls": val_recalls,
        "val_f1s": val_f1s,
    }
    with open("training_metrics.json", "w") as f:
        json.dump(metrics, f)



#*---------------------------- Print epoch metrics ----------------------------
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.5f}, Acc: {train_acc*100:.2f}%, Precision: {train_precision:.4f}, Recall: {train_recall:.4f}, F1: {train_f1:.4f}")
    print(f"Val   Loss: {val_loss:.5f}, Acc: {val_acc*100:.2f}%, Precision: {val_precision:.4f}, Recall: {val_recall:.4f}, F1: {val_f1:.4f}")
    print("-----------------------------------------------------------")

end_time = timer() # end timing
training_time = print_train_time(start_time,end_time, device)

# Section 3

In [ ]:
class_names = train_dataset.classes
print(f"Class names: {class_names}")

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(train_accuracies, label="Train Accuracy")
plt.plot(val_accuracies, label="Val Accuracy")
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig("/kaggle/working/Acc_MRI_RN18_M1_TL.png") 
plt.show()

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig("/kaggle/working/Loss_MRI_RN18_M1_TL.png") 

plt.show()

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(train_precisions, label="Train Precision")
plt.plot(val_precisions, label="Val Precision")
plt.xlabel('Epochs')
plt.ylabel('Precision')
plt.title('Precision')
plt.legend()
plt.grid(True)
# 

plt.tight_layout()
plt.savefig("/kaggle/working/Precision_MRI_RN18_M1_TL.png") 

plt.show()


In [ ]:
plt.figure(figsize=(8,6))
plt.plot(train_recalls, label="Train Recall")
plt.plot(val_recalls, label="Val Recall")
plt.xlabel('Epochs')
plt.ylabel('Recall')
plt.title('Recall')
plt.legend()
plt.grid(True)


plt.tight_layout()
plt.savefig("/kaggle/working/Recall_MRI_RN18_M1_TL.png") 

plt.show()

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(train_f1s, label="Train F1")
plt.plot(val_f1s, label="Val F1")
plt.xlabel('Epochs')
plt.ylabel('F1 Score')
plt.title('F1 Score')
plt.legend()
plt.grid(True)


plt.tight_layout()
plt.savefig("/kaggle/working/F1-Score_MRI_RN18_M1_TL.png") 

plt.show()

In [ ]:
# Define the paths to save the model
save_path_state_dict = "state_dict_MRI_RN18_M1_TL.pth"
save_path_entire_model = "entire_MRI_RN18_M1_TL.pth"

# 1. Save the model's state_dict (recommended way for inference and transfer learning)
torch.save(model.state_dict(), save_path_state_dict)
print(f"Model state dictionary saved to {save_path_state_dict}")

# 2. Save the entire model (includes model architecture and state_dict)
# Note: This method is less flexible for transfer learning or if the model
# definition changes, as you need the exact model class defined when loading.
torch.save(model, save_path_entire_model)
print(f"Entire model saved to {save_path_entire_model}")